In [40]:
import cv2
import mediapipe as mp
import numpy as np

In [41]:
def euclidean_distance(x1, x2):
    total = 0
    for i in range(1, len(x1)):
        total += (x1[i]-x2[i])**2
    return total ** 0.5

In [42]:
def stateDistance(vector):
    if len(vector) == 3:
        a1 = euclidean_distance(vector[0], vector[1])
        a2 = euclidean_distance(vector[1], vector[2])
        return [a1, a2]
    if len(vector) == 4:
        a1 = euclidean_distance(vector[0], vector[1])
        a2 = euclidean_distance(vector[1], vector[2])
        a3 = euclidean_distance(vector[2], vector[3])
        return [a1, a2, a3]

In [ ]:
class HandDetector:
    def __init__(self, mode=False, maxHands=1, modelC=1, detectionCon=0.5, trackCon=0.5):
        self.mode = mode
        self.maxHands = maxHands
        self.detectionCon = detectionCon
        self.modelC = modelC
        self.trackCon = trackCon

        self.mpHands = mp.solutions.hands
        self.hands = self.mpHands.Hands(self.mode, self.maxHands, self.modelC, self.detectionCon, self.trackCon)
        self.mpDraw = mp.solutions.drawing_utils

    def findHands(self, img):
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(imgRGB)

        if self.results.multi_hand_landmarks:
            for handLandmarks in self.results.multi_hand_landmarks:
                self.mpDraw.draw_landmarks(img, handLandmarks, self.mpHands.HAND_CONNECTIONS,
                    self.mpDraw.DrawingSpec(color=(255,0,0), thickness=2, circle_radius=2),
                    self.mpDraw.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2))
        return img

    def findPosition(self, img):
        landmarkList = []
        if self.results.multi_hand_landmarks:
            myHand = self.results.multi_hand_landmarks[0]
            for id, lm in enumerate(myHand.landmark):
                h, w, c = img.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                landmarkList.append([id, cx, cy])
        return landmarkList

    def asl_alphabet_recognizer(self, landmarkList):
        """Recognizes ASL alphabet letters A-Z from MediaPipe hand landmarks.
        Designed for a right hand with palm facing the camera.
        Returns the detected letter, or "" if unrecognized.

        Landmark reference (MediaPipe 21-point hand model):
          0=Wrist, 1-4=Thumb(CMC,MCP,IP,Tip), 5-8=Index(MCP,PIP,DIP,Tip),
          9-12=Middle(MCP,PIP,DIP,Tip), 13-16=Ring(MCP,PIP,DIP,Tip), 17-20=Pinky(MCP,PIP,DIP,Tip)
        """
        if len(landmarkList) == 0:
            return ""

        lm = landmarkList

        # --- Finger state detection ---
        # Thumb: compare x-coordinates (right hand: tip moves left when extended)
        thumbState = ""
        if lm[2][1] > lm[3][1] > lm[4][1]:
            thumbState = "CLOSE"
        elif lm[2][1] < lm[3][1] < lm[4][1]:
            thumbState = "OPEN"

        # Fingers: compare y-coordinates (smaller y = higher in image = extended)
        # OPEN: tip above MCP,  CLOSE: tip below MCP,  HALF_OPEN: tip above PIP but PIP below MCP
        def get_state(mcp, pip, tip):
            if lm[mcp][2] > lm[pip][2] > lm[tip][2]:
                return "OPEN"
            elif lm[mcp][2] < lm[pip][2] < lm[tip][2]:
                return "CLOSE"
            elif lm[mcp][2] > lm[pip][2] and lm[tip][2] > lm[pip][2]:
                return "HALF_OPEN"
            return ""

        indexState  = get_state(6, 7, 8)
        middleState = get_state(10, 11, 12)
        ringState   = get_state(14, 15, 16)
        pinkyState  = get_state(18, 19, 20)

        # --- Reference distances ---
        ref   = euclidean_distance(lm[0], lm[1])   # wrist to thumb CMC (scale reference)
        t_len = euclidean_distance(lm[3], lm[4])   # thumb distal phalanx length

        d_t_i = euclidean_distance(lm[4], lm[8])   # thumb tip <-> index tip
        d_t_m = euclidean_distance(lm[4], lm[12])  # thumb tip <-> middle tip
        d_t_r = euclidean_distance(lm[4], lm[16])  # thumb tip <-> ring tip
        d_t_p = euclidean_distance(lm[4], lm[20])  # thumb tip <-> pinky tip
        d_i_m = euclidean_distance(lm[8], lm[12])  # index tip <-> middle tip
        d_m_r = euclidean_distance(lm[12], lm[16]) # middle tip <-> ring tip
        d_r_p = euclidean_distance(lm[16], lm[20]) # ring tip <-> pinky tip

        # Is index finger pointing horizontally? (G, H detection)
        index_dx = abs(lm[8][1] - lm[6][1])
        index_dy = abs(lm[8][2] - lm[6][2])
        index_horiz = index_dx > index_dy

        # Is middle finger pointing horizontally alongside index? (H detection)
        middle_dx = abs(lm[12][1] - lm[10][1])
        middle_dy = abs(lm[12][2] - lm[10][2])
        middle_horiz = middle_dx > middle_dy

        letter = ""

        # =====================================================================
        # ASL Alphabet Classification
        # =====================================================================

        # --- B: All 4 fingers open, thumb tucked ---
        if indexState == "OPEN" and middleState == "OPEN" and ringState == "OPEN" and pinkyState == "OPEN":
            letter = "B"

        # --- W: Index + Middle + Ring open, Pinky closed ---
        elif indexState == "OPEN" and middleState == "OPEN" and ringState == "OPEN" and pinkyState == "CLOSE":
            letter = "W"

        # --- Y: Thumb + Pinky extended, others closed ---
        elif (thumbState == "OPEN" and indexState == "CLOSE" and middleState == "CLOSE"
              and ringState == "CLOSE" and pinkyState == "OPEN"):
            letter = "Y"

        # --- L: Index + Thumb extended (L shape), others closed ---
        elif (thumbState == "OPEN" and indexState == "OPEN" and middleState == "CLOSE"
              and ringState == "CLOSE" and pinkyState == "CLOSE"):
            letter = "L"

        # --- I / J: Only pinky extended (J adds a motion, same handshape) ---
        elif (indexState == "CLOSE" and middleState == "CLOSE"
              and ringState == "CLOSE" and pinkyState == "OPEN"):
            letter = "I"

        # --- Index + Middle open, Ring + Pinky closed: V, U, R, K, H ---
        elif indexState == "OPEN" and middleState == "OPEN" and ringState == "CLOSE" and pinkyState == "CLOSE":
            if index_horiz and middle_horiz:
                letter = "H"                    # two fingers pointing sideways
            elif thumbState == "OPEN" and d_t_m < 1.5 * ref:
                letter = "K"                    # thumb between index and middle
            elif d_i_m > 1.5 * ref:
                letter = "V"                    # fingers spread in V
            elif abs(lm[8][1] - lm[12][1]) < 0.4 * ref:
                letter = "R"                    # fingers crossed
            else:
                letter = "U"                    # fingers together pointing up

        # --- Only Index open: D or G ---
        elif indexState == "OPEN" and middleState == "CLOSE" and ringState == "CLOSE" and pinkyState == "CLOSE":
            if index_horiz and thumbState == "OPEN":
                letter = "G"                    # index + thumb pointing sideways (gun shape)
            elif d_t_m < 1.2 * ref or d_t_r < 1.2 * ref:
                letter = "D"                    # index up, thumb makes O with middle/ring
            else:
                letter = "D"

        # --- Middle + Ring + Pinky open, Index closed: F ---
        elif (indexState in ["CLOSE", "HALF_OPEN"] and middleState == "OPEN"
              and ringState == "OPEN" and pinkyState == "OPEN"):
            letter = "F"                        # index + thumb pinch, other 3 fingers up

        # --- Index hooked (HALF_OPEN), others closed: X ---
        elif (indexState == "HALF_OPEN" and middleState == "CLOSE"
              and ringState == "CLOSE" and pinkyState == "CLOSE"):
            letter = "X"

        # --- All fingers HALF_OPEN (curved): C or O ---
        elif (indexState == "HALF_OPEN" and middleState == "HALF_OPEN"
              and ringState == "HALF_OPEN" and pinkyState == "HALF_OPEN"):
            if d_t_i < 1.2 * ref:
                letter = "O"                    # fingers touching thumb (closed O)
            else:
                letter = "C"                    # fingers curved but open (C shape)

        # --- Index + Middle HALF_OPEN, Ring + Pinky closed: N ---
        elif (indexState == "HALF_OPEN" and middleState == "HALF_OPEN"
              and ringState == "CLOSE" and pinkyState == "CLOSE"):
            letter = "N"

        # --- Index + Middle + Ring HALF_OPEN, Pinky closed: M ---
        elif (indexState == "HALF_OPEN" and middleState == "HALF_OPEN"
              and ringState == "HALF_OPEN" and pinkyState == "CLOSE"):
            letter = "M"

        # --- All fingers CLOSE (fist variants): A, E, S, T ---
        elif (indexState == "CLOSE" and middleState == "CLOSE"
              and ringState == "CLOSE" and pinkyState == "CLOSE"):
            if thumbState == "OPEN":
                letter = "A"                    # fist with thumb to the side
            elif d_t_i < 0.8 * ref:
                letter = "S"                    # thumb over fingers (thumb tip near index)
            elif d_t_m < 1.0 * ref:
                letter = "T"                    # thumb tucked between index and middle
            else:
                letter = "E"                    # fingers bent/curled, thumb tucked under

        return letter


In [ ]:
if __name__ == "__main__":
    cap = cv2.VideoCapture(0)
    detector = HandDetector()

    asl_letters = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

    while True:
        success, img = cap.read()
        img = detector.findHands(img)
        landmarkList = detector.findPosition(img)

        if len(landmarkList) != 0:
            letter = detector.asl_alphabet_recognizer(landmarkList)
            if letter:
                cv2.putText(img, letter,
                            (landmarkList[0][1] - 50, landmarkList[0][2] - 80),
                            cv2.FONT_HERSHEY_PLAIN, 5, (0, 0, 255), 4)

        cv2.imshow("ASL Alphabet Recognition", img)

        if cv2.waitKey(1) == ord('q'):
            break

    cv2.destroyAllWindows()
    cap.release()
